# 04 - Gold Layer Refinement

## General Solution
- Schema to be used: Star
    - pros
        - Simpler Design
        - Faster Queries - fewer joins
    - cons
        - Data Redundancy
        

### Modeling
- Fact table: fact_orders
- Dimension tables:
    - dim_customer - Flattened customer and location attributes
    - dim_time - Flattened date attributes including time periods. 
        - This dimension will be used by other facts

### Data Ingestion Script

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;
SET TIME ZONE 'America/Sao_Paulo';

-- dim_customer
-- Flattened customer dimension with location attributes
CREATE OR REPLACE TABLE dim_customer AS
SELECT
  ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
  customer_id,
  customer_unique_id,
  INITCAP(customer_city) AS customer_city,
  customer_state,
  CONCAT(INITCAP(customer_city), ' - ', customer_state) AS customer_city_state
FROM `02-silver`.customer;

-- Primary key constraint (ROW_NUMBER() produces nullable; enforce NOT NULL first)
ALTER TABLE dim_customer ALTER COLUMN customer_key SET NOT NULL;
ALTER TABLE dim_customer ADD CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key);

-- Table and column comments
COMMENT ON TABLE dim_customer IS 'Flattened customer dimension with location attributes (city, state). Surrogate key: customer_key. Natural key: customer_id.';
COMMENT ON COLUMN dim_customer.customer_key IS 'Surrogate primary key generated via ROW_NUMBER().';
COMMENT ON COLUMN dim_customer.customer_id IS 'Natural key from source. Unique per order transaction (one customer_id per order).';
COMMENT ON COLUMN dim_customer.customer_unique_id IS 'Business key identifying the real customer entity. Multiple customer_id values can map to the same customer_unique_id.';
COMMENT ON COLUMN dim_customer.customer_city IS 'Customer city name, title-cased for readability.';
COMMENT ON COLUMN dim_customer.customer_state IS 'Brazilian state code (two-letter abbreviation).';
COMMENT ON COLUMN dim_customer.customer_city_state IS 'Concatenated city and state for convenient grouping and display.';

-- Verify: Gold vs Silver row count reconciliation
SELECT
  'dim_customer' AS gold_table_name,
  COUNT(*) AS gold_count,
  (SELECT COUNT(*) FROM `02-silver`.customer) AS silver_count,
  CASE
    WHEN COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.customer) THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS reconciliation_status
FROM dim_customer;






In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;
SET TIME ZONE 'America/Sao_Paulo';


-- Flattened date attributes including time periods
-- Generated from the full date range of order_purchase_timestamp.
-- This dimension will be used by other facts
-- ref https://knowledge.bigenius-x.com/create-a-date-dimension

CREATE OR REPLACE TABLE dim_time AS
WITH min_max_order_dates AS (
  SELECT
    CAST(MIN(order_purchase_timestamp) AS DATE) AS min_date,
    CAST(MAX(order_purchase_timestamp) AS DATE) AS max_date
  FROM `02-silver`.order
),
date_series AS (
  SELECT explode(sequence(min_date, max_date, INTERVAL 1 DAY)) AS full_date
  FROM min_max_order_dates
),
-- Brazilian national and observed holidays (fixed and variable dates)
brazil_holidays AS (
  SELECT CAST('2016-01-01' AS DATE) AS holiday_date, 'Confraternização Universal' AS holiday_name
  UNION ALL SELECT CAST('2016-02-09' AS DATE), 'Carnaval (Terça-feira)'
  UNION ALL SELECT CAST('2016-03-25' AS DATE), 'Sexta-feira Santa'
  UNION ALL SELECT CAST('2016-05-26' AS DATE), 'Corpus Christi'
  UNION ALL SELECT CAST('2016-04-21' AS DATE), 'Tiradentes'
  UNION ALL SELECT CAST('2016-05-01' AS DATE), 'Dia do Trabalho'
  UNION ALL SELECT CAST('2016-09-07' AS DATE), 'Independência do Brasil'
  UNION ALL SELECT CAST('2016-10-12' AS DATE), 'Nossa Senhora Aparecida'
  UNION ALL SELECT CAST('2016-11-02' AS DATE), 'Finados'
  UNION ALL SELECT CAST('2016-11-15' AS DATE), 'Proclamação da República'
  UNION ALL SELECT CAST('2016-12-25' AS DATE), 'Natal'
  UNION ALL SELECT CAST('2017-01-01' AS DATE), 'Confraternização Universal'
  UNION ALL SELECT CAST('2017-02-28' AS DATE), 'Carnaval (Terça-feira)'
  UNION ALL SELECT CAST('2017-04-14' AS DATE), 'Sexta-feira Santa'
  UNION ALL SELECT CAST('2017-06-15' AS DATE), 'Corpus Christi'
  UNION ALL SELECT CAST('2017-04-21' AS DATE), 'Tiradentes'
  UNION ALL SELECT CAST('2017-05-01' AS DATE), 'Dia do Trabalho'
  UNION ALL SELECT CAST('2017-09-07' AS DATE), 'Independência do Brasil'
  UNION ALL SELECT CAST('2017-10-12' AS DATE), 'Nossa Senhora Aparecida'
  UNION ALL SELECT CAST('2017-11-02' AS DATE), 'Finados'
  UNION ALL SELECT CAST('2017-11-15' AS DATE), 'Proclamação da República'
  UNION ALL SELECT CAST('2017-12-25' AS DATE), 'Natal'
  UNION ALL SELECT CAST('2018-01-01' AS DATE), 'Confraternização Universal'
  UNION ALL SELECT CAST('2018-02-13' AS DATE), 'Carnaval (Terça-feira)'
  UNION ALL SELECT CAST('2018-03-30' AS DATE), 'Sexta-feira Santa'
  UNION ALL SELECT CAST('2018-05-31' AS DATE), 'Corpus Christi'
  UNION ALL SELECT CAST('2018-04-21' AS DATE), 'Tiradentes'
  UNION ALL SELECT CAST('2018-05-01' AS DATE), 'Dia do Trabalho'
  UNION ALL SELECT CAST('2018-09-07' AS DATE), 'Independência do Brasil'
  UNION ALL SELECT CAST('2018-10-12' AS DATE), 'Nossa Senhora Aparecida'
  UNION ALL SELECT CAST('2018-11-02' AS DATE), 'Finados'
  UNION ALL SELECT CAST('2018-11-15' AS DATE), 'Proclamação da República'
  UNION ALL SELECT CAST('2018-12-25' AS DATE), 'Natal'
)
SELECT
  CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
  full_date,
  YEAR(full_date) AS year,
  QUARTER(full_date) AS quarter,
  CASE WHEN MONTH(full_date) <= 6 THEN 1 ELSE 2 END AS semester,
  CONCAT('Q', QUARTER(full_date), ' ', YEAR(full_date)) AS quarter_name,
  MONTH(full_date) AS month,
  date_format(full_date, 'MMMM') AS month_name,
  date_format(full_date, 'yyyy-MM') AS year_month,
  DAY(full_date) AS day_of_month,
  DAYOFWEEK(full_date) AS day_of_week,
  date_format(full_date, 'EEEE') AS day_name,
  DAYOFWEEK(full_date) IN (1, 7) AS is_weekend,
  bh.holiday_name IS NOT NULL AS is_holiday,
  bh.holiday_name AS holiday_name
FROM date_series ds
LEFT JOIN brazil_holidays bh ON ds.full_date = bh.holiday_date;

ALTER TABLE dim_time ALTER COLUMN date_key SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN full_date SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN year SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN quarter SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN semester SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN quarter_name SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN month SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN month_name SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN year_month SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN day_of_month SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN day_of_week SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN day_name SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN is_weekend SET NOT NULL;
ALTER TABLE dim_time ALTER COLUMN is_holiday SET NOT NULL;

ALTER TABLE dim_time ADD CONSTRAINT pk_dim_time PRIMARY KEY (date_key);

-- Table and column comments
COMMENT ON TABLE dim_time IS 'Conformed time dimension with flattened date attributes. date_key is YYYYMMDD integer. Covers the full date range of order data. Reusable by other fact tables.';
COMMENT ON COLUMN dim_time.date_key IS 'Surrogate primary key in YYYYMMDD integer format (e.g., 20180101 for 2018-01-01).';
COMMENT ON COLUMN dim_time.full_date IS 'The actual DATE value.';
COMMENT ON COLUMN dim_time.year IS 'Calendar year (e.g., 2018).';
COMMENT ON COLUMN dim_time.quarter IS 'Calendar quarter number (1–4).';
COMMENT ON COLUMN dim_time.semester IS 'Calendar semester: 1 for Jan–Jun, 2 for Jul–Dec.';
COMMENT ON COLUMN dim_time.quarter_name IS 'Human-readable quarter label (e.g., Q1 2018).';
COMMENT ON COLUMN dim_time.month IS 'Calendar month number (1–12).';
COMMENT ON COLUMN dim_time.month_name IS 'Full month name (e.g., January).';
COMMENT ON COLUMN dim_time.year_month IS 'Year-month string in yyyy-MM format (e.g., 2018-01).';
COMMENT ON COLUMN dim_time.day_of_month IS 'Day of the month (1–31).';
COMMENT ON COLUMN dim_time.day_of_week IS 'Day of the week number per Spark convention (1=Sunday, 7=Saturday).';
COMMENT ON COLUMN dim_time.day_name IS 'Full day-of-week name (e.g., Monday).';
COMMENT ON COLUMN dim_time.is_weekend IS 'TRUE if the date falls on Saturday (7) or Sunday (1).';
COMMENT ON COLUMN dim_time.is_holiday IS 'TRUE if the date is a Brazilian national or observed holiday.';
COMMENT ON COLUMN dim_time.holiday_name IS 'Name of the Brazilian holiday, or NULL if not a holiday.';


-- Verify
SELECT COUNT(*) AS dim_time_row_count, MIN(full_date) AS min_date, MAX(full_date) AS max_date FROM dim_time;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;
SET TIME ZONE 'America/Sao_Paulo';

-- fact_orders - Fact table at order grain (one row per order)
-- Measures from order_item and order_payment

-- Temp view order items (price, freight, item count per order)
CREATE OR REPLACE TEMP VIEW v_order_items_agg AS
SELECT
  order_id,
  COUNT(*) AS total_item_count,
  SUM(price) AS total_price,
  SUM(freight_value) AS total_freight_value
  FROM `02-silver`.order_item
GROUP BY order_id;

-- Temp view order payments (payment value, payment count per order)
CREATE OR REPLACE TEMP VIEW v_order_payments_agg AS
SELECT
  order_id,
  COUNT(*) AS payment_count,
  SUM(payment_value) AS total_payment_value
FROM `02-silver`.order_payment
GROUP BY order_id;

-- Temp view order reviews (latest review score per order)
CREATE OR REPLACE TEMP VIEW v_order_review_agg AS
SELECT
  order_id,
  MAX(review_score) AS review_score
FROM `02-silver`.order_reviews
GROUP BY order_id;

-- Building the fact table
CREATE OR REPLACE TABLE fact_orders AS
SELECT
  ROW_NUMBER() OVER (ORDER BY o.order_id) AS order_key,
  o.order_id,
  dc.customer_key,
  CAST(date_format(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) AS purchase_date_key,
  o.order_status_id,
  os.order_status_name,
  o.order_purchase_timestamp,
  o.order_approved_at,
  o.order_delivered_carrier_date,
  o.order_delivered_customer_date,
  o.order_estimated_delivery_date,
  COALESCE(oi.total_item_count, 0) AS total_item_count,
  COALESCE(oi.total_price, 0) AS total_price,
  COALESCE(oi.total_freight_value, 0) AS total_freight_value,
  COALESCE(oi.total_price, 0) + COALESCE(oi.total_freight_value, 0) AS total_order_value,
  COALESCE(op.total_payment_value, 0) AS total_payment_value,
  COALESCE(op.payment_count, 0) AS payment_count,
  DATEDIFF(o.order_estimated_delivery_date, o.order_purchase_timestamp) AS estimated_delivery_days,
  DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp) AS actual_delivery_days,
  rv.review_score,
  CASE WHEN rv.review_score >= 4 THEN 'good' WHEN rv.review_score <= 2 THEN 'bad' ELSE 'neutral' END AS review_classification,
  CASE WHEN rv.review_score >= 4 THEN TRUE ELSE FALSE END AS is_good_review,
  CASE WHEN DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date) > 0 THEN TRUE ELSE FALSE END AS is_delivery_delayed
  
FROM `02-silver`.order o
LEFT JOIN v_order_items_agg oi ON o.order_id = oi.order_id
LEFT JOIN v_order_payments_agg op ON o.order_id = op.order_id
LEFT JOIN `02-silver`.order_status os ON o.order_status_id = os.order_status_id
LEFT JOIN dim_customer dc ON o.customer_id = dc.customer_id
LEFT JOIN v_order_review_agg rv ON o.order_id = rv.order_id;

ALTER TABLE fact_orders ALTER COLUMN order_key SET NOT NULL;
ALTER TABLE fact_orders ALTER COLUMN customer_key SET NOT NULL;
ALTER TABLE fact_orders ALTER COLUMN purchase_date_key SET NOT NULL;

ALTER TABLE fact_orders ADD CONSTRAINT pk_fact_orders PRIMARY KEY (order_key);
ALTER TABLE fact_orders ADD CONSTRAINT fk_fact_orders_customer
  FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key);
ALTER TABLE fact_orders ADD CONSTRAINT fk_fact_orders_time
  FOREIGN KEY (purchase_date_key) REFERENCES dim_time(date_key);

COMMENT ON TABLE fact_orders IS 'Fact table at order grain. One row per order. Measures aggregated from order items and payments. FKs to dim_customer and dim_time.';
COMMENT ON COLUMN fact_orders.order_key IS 'Surrogate primary key.';
COMMENT ON COLUMN fact_orders.order_id IS 'Degenerate dimension (natural key from source order).';
COMMENT ON COLUMN fact_orders.customer_key IS 'Foreign key to dim_customer.';
COMMENT ON COLUMN fact_orders.purchase_date_key IS 'Foreign key to dim_time, derived from order_purchase_timestamp date.';
COMMENT ON COLUMN fact_orders.total_order_value IS 'Sum of item prices and freight values for the order.';
COMMENT ON COLUMN fact_orders.total_payment_value IS 'Sum of all payment values for the order.';
COMMENT ON COLUMN fact_orders.estimated_delivery_days IS 'Days between order purchase and estimated delivery date.';
COMMENT ON COLUMN fact_orders.actual_delivery_days IS 'Days between order purchase and actual customer delivery date. NULL if not yet delivered.';
COMMENT ON COLUMN fact_orders.review_score IS 'Customer review score (1-5) from the source order_review table. NULL if no review submitted.';

COMMENT ON COLUMN fact_orders.review_classification IS 'Categorical review label: good (score >= 4), bad (score <= 2), neutral (score = 3). NULL if no review.';
COMMENT ON COLUMN fact_orders.is_good_review IS 'TRUE if review score is 4 or 5 (good), FALSE otherwise (including no review).';
COMMENT ON COLUMN fact_orders.order_status_id IS 'Foreign key to the order_status dimension-like table (source natural key for order status).';
COMMENT ON COLUMN fact_orders.order_status_name IS 'Human-readable order status name (e.g., delivered, shipped, canceled).';
COMMENT ON COLUMN fact_orders.order_purchase_timestamp IS 'Timestamp when the order was purchased by the customer.';
COMMENT ON COLUMN fact_orders.order_approved_at IS 'Timestamp when the order payment was approved. NULL if not yet approved.';
COMMENT ON COLUMN fact_orders.order_delivered_carrier_date IS 'Timestamp when the order was handed to the logistics partner. NULL if not yet handed off.';
COMMENT ON COLUMN fact_orders.order_delivered_customer_date IS 'Timestamp when the order was delivered to the customer. NULL if not yet delivered.';
COMMENT ON COLUMN fact_orders.order_estimated_delivery_date IS 'Estimated delivery date promised to the customer at purchase time.';
COMMENT ON COLUMN fact_orders.total_item_count IS 'Number of items in the order (count of order_item rows).';
COMMENT ON COLUMN fact_orders.total_price IS 'Sum of all item prices in the order (R$).';
COMMENT ON COLUMN fact_orders.total_freight_value IS 'Sum of all freight values in the order (R$).';
COMMENT ON COLUMN fact_orders.payment_count IS 'Number of payment records for the order.';
COMMENT ON COLUMN fact_orders.is_delivery_delayed IS 'TRUE if the order was delivered after the estimated delivery date, FALSE otherwise. NULL if not yet delivered.';

-- Verify
SELECT COUNT(*) AS fact_orders_row_count FROM fact_orders;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;

-- Reconciliation: Gold vs Silver row counts
SELECT
  'fact_orders' AS gold_table_name,
  (SELECT COUNT(*) FROM fact_orders) AS gold_count,
  (SELECT COUNT(*) FROM `02-silver`.order) AS silver_count,
  (SELECT COUNT(*) FROM fact_orders) - (SELECT COUNT(*) FROM `02-silver`.order) AS gold_minus_silver,
  CASE
    WHEN (SELECT COUNT(*) FROM fact_orders) = (SELECT COUNT(*) FROM `02-silver`.order) THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS reconciliation_status
UNION ALL
SELECT
  'dim_customer' AS gold_table_name,
  (SELECT COUNT(*) FROM dim_customer) AS gold_count,
  (SELECT COUNT(*) FROM `02-silver`.customer) AS silver_count,
  (SELECT COUNT(*) FROM dim_customer) - (SELECT COUNT(*) FROM `02-silver`.customer) AS gold_minus_silver,
  CASE
    WHEN (SELECT COUNT(*) FROM dim_customer) = (SELECT COUNT(*) FROM `02-silver`.customer) THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS reconciliation_status;

## Additional Fact Tables Using Conformed dim_time

The `dim_time` dimension is **conformed** — shared across multiple fact tables so they can be analyzed together on a common calendar:

- **fact_order_items** — Order-item grain. FKs to `dim_customer`, `dim_time` (purchase date + shipping date), `dim_product`.
- **fact_order_payments** — Payment grain. FKs to `dim_customer`, `dim_time` (purchase date), `dim_payment_type`.

New dimensions:
- **dim_product** — Flattened from `02-silver.product` + `02-silver.category`
- **dim_payment_type** — Flattened from `02-silver.payment`

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;
SET TIME ZONE 'America/Sao_Paulo';

-- ============================================================
-- dim_product
-- Flattened product dimension with category name and physical attributes.
-- Sources: 02-silver.product, 02-silver.category
-- ============================================================
CREATE OR REPLACE TABLE dim_product AS
SELECT
  ROW_NUMBER() OVER (ORDER BY p.product_id) AS product_key,
  p.product_id,
  p.category_id,
  c.product_category_name,
  upper(c.product_category_name) AS product_category_name_clean
  
FROM `02-silver`.product p
LEFT JOIN `02-silver`.category c ON p.category_id = c.category_id;

ALTER TABLE dim_product ALTER COLUMN product_key SET NOT NULL;
ALTER TABLE dim_product ADD CONSTRAINT pk_dim_product PRIMARY KEY (product_key);

COMMENT ON TABLE dim_product IS 'Flattened product dimension with category name and physical attributes. Surrogate key: product_key. Natural key: product_id.';
COMMENT ON COLUMN dim_product.product_key IS 'Surrogate primary key generated via ROW_NUMBER().';
COMMENT ON COLUMN dim_product.product_id IS 'Natural key from source product table.';
COMMENT ON COLUMN dim_product.category_id IS 'Foreign key to the category dimension-like table (source natural key for product category).';
COMMENT ON COLUMN dim_product.product_category_name IS 'Original category name from source (lowercase, underscore-separated).';
COMMENT ON COLUMN dim_product.product_category_name_clean IS 'Category name title-cased for readability.';


-- Verify
SELECT COUNT(*) AS dim_product_row_count FROM dim_product;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;
SET TIME ZONE 'America/Sao_Paulo';

-- dim_payment_type - Payment type dimension.
CREATE OR REPLACE TABLE dim_payment_type AS
SELECT
  ROW_NUMBER() OVER (ORDER BY payment_type_id) AS payment_type_key,
  payment_type_id,
  payment_type_name,
  INITCAP(REPLACE(payment_type_name, '_', ' ')) AS payment_type_name_clean
FROM `02-silver`.payment;

ALTER TABLE dim_payment_type ALTER COLUMN payment_type_key SET NOT NULL;
ALTER TABLE dim_payment_type ADD CONSTRAINT pk_dim_payment_type PRIMARY KEY (payment_type_key);

COMMENT ON TABLE dim_payment_type IS 'Payment type dimension. Surrogate key: payment_type_key. Natural key: payment_type_id.';
COMMENT ON COLUMN dim_payment_type.payment_type_key IS 'Surrogate primary key generated via ROW_NUMBER().';
COMMENT ON COLUMN dim_payment_type.payment_type_id IS 'Natural key from source payment table.';
COMMENT ON COLUMN dim_payment_type.payment_type_name IS 'Original payment type name from source.';
COMMENT ON COLUMN dim_payment_type.payment_type_name_clean IS 'Payment type name title-cased and underscore-replaced for readability.';


-- Verify
SELECT * FROM dim_payment_type;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;
SET TIME ZONE 'America/Sao_Paulo';

-- fact_order_items - Fact table at order-item grain (one row per order item).
CREATE OR REPLACE TABLE fact_order_items AS
SELECT
  ROW_NUMBER() OVER (ORDER BY oi.order_id, oi.order_item_id) AS order_item_fact_key,
  oi.order_id,
  oi.order_item_id,
  dc.customer_key,
  CAST(date_format(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) AS purchase_date_key,
  CAST(date_format(oi.shipping_limit_date, 'yyyyMMdd') AS INT) AS shipping_date_key,
  dp.product_key,
  oi.price,
  oi.freight_value,
  oi.price + oi.freight_value AS total_value
FROM `02-silver`.order_item oi
INNER JOIN `02-silver`.order o ON oi.order_id = o.order_id
INNER JOIN dim_customer dc ON o.customer_id = dc.customer_id
INNER JOIN dim_product dp ON oi.product_id = dp.product_id;

ALTER TABLE fact_order_items ALTER COLUMN order_item_fact_key SET NOT NULL;
ALTER TABLE fact_order_items ALTER COLUMN customer_key SET NOT NULL;
ALTER TABLE fact_order_items ALTER COLUMN purchase_date_key SET NOT NULL;
ALTER TABLE fact_order_items ALTER COLUMN shipping_date_key SET NOT NULL;
ALTER TABLE fact_order_items ALTER COLUMN product_key SET NOT NULL;

ALTER TABLE fact_order_items ADD CONSTRAINT pk_fact_order_items PRIMARY KEY (order_item_fact_key);
ALTER TABLE fact_order_items ADD CONSTRAINT fk_foi_dim_customer
  FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key);
ALTER TABLE fact_order_items ADD CONSTRAINT fk_foi_dim_time_purchase
  FOREIGN KEY (purchase_date_key) REFERENCES dim_time(date_key);
ALTER TABLE fact_order_items ADD CONSTRAINT fk_foi_dim_time_shipping
  FOREIGN KEY (shipping_date_key) REFERENCES dim_time(date_key);
ALTER TABLE fact_order_items ADD CONSTRAINT fk_foi_dim_product
  FOREIGN KEY (product_key) REFERENCES dim_product(product_key);

COMMENT ON TABLE fact_order_items IS 'Fact table at order-item grain. One row per order item. FKs to dim_customer, dim_time (purchase + shipping dates), dim_product. Measures: price, freight_value, total_value.';
COMMENT ON COLUMN fact_order_items.order_item_fact_key IS 'Surrogate primary key.';
COMMENT ON COLUMN fact_order_items.order_id IS 'Degenerate dimension (natural key from source order).';
COMMENT ON COLUMN fact_order_items.order_item_id IS 'Degenerate dimension (sequence number of item within order).';
COMMENT ON COLUMN fact_order_items.purchase_date_key IS 'FK to dim_time — date the order was purchased.';
COMMENT ON COLUMN fact_order_items.shipping_date_key IS 'FK to dim_time — shipping limit date for the item.';
COMMENT ON COLUMN fact_order_items.total_value IS 'price + freight_value for this order item.';

SELECT COUNT(*) AS fact_order_items_row_count FROM fact_order_items;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;
SET TIME ZONE 'America/Sao_Paulo';

-- fact_order_payments
-- Fact table at payment grain (one row per payment record).
-- FKs: dim_customer, dim_time (purchase_date_key), dim_payment_type
-- Sources: 02-silver.order_payment, order, dim_customer, dim_time, dim_payment_type

CREATE OR REPLACE TABLE fact_order_payments AS
SELECT
  ROW_NUMBER() OVER (ORDER BY op.order_id, op.payment_sequential) AS order_payment_fact_key,
  op.order_id,
  op.payment_sequential,
  dc.customer_key,
  CAST(date_format(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) AS purchase_date_key,
  dpt.payment_type_key,
  op.payment_installments,
  op.payment_value
FROM `02-silver`.order_payment op
INNER JOIN `02-silver`.order o ON op.order_id = o.order_id
INNER JOIN dim_customer dc ON o.customer_id = dc.customer_id
INNER JOIN dim_payment_type dpt ON op.payment_type_id = dpt.payment_type_id;

-- Constraints (enforce NOT NULL on key columns before adding constraints)
ALTER TABLE fact_order_payments ALTER COLUMN order_payment_fact_key SET NOT NULL;
ALTER TABLE fact_order_payments ALTER COLUMN customer_key SET NOT NULL;
ALTER TABLE fact_order_payments ALTER COLUMN purchase_date_key SET NOT NULL;
ALTER TABLE fact_order_payments ALTER COLUMN payment_type_key SET NOT NULL;

ALTER TABLE fact_order_payments ADD CONSTRAINT pk_fact_order_payments PRIMARY KEY (order_payment_fact_key);
ALTER TABLE fact_order_payments ADD CONSTRAINT fk_fop_dim_customer
  FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key);
ALTER TABLE fact_order_payments ADD CONSTRAINT fk_fop_dim_time
  FOREIGN KEY (purchase_date_key) REFERENCES dim_time(date_key);
ALTER TABLE fact_order_payments ADD CONSTRAINT fk_fop_dim_payment_type
  FOREIGN KEY (payment_type_key) REFERENCES dim_payment_type(payment_type_key);

-- Table and column comments
COMMENT ON TABLE fact_order_payments IS 'Fact table at payment grain. One row per payment record. FKs to dim_customer, dim_time (purchase date), dim_payment_type. Measures: payment_value, payment_installments.';
COMMENT ON COLUMN fact_order_payments.order_payment_fact_key IS 'Surrogate primary key.';
COMMENT ON COLUMN fact_order_payments.order_id IS 'Degenerate dimension (natural key from source order).';
COMMENT ON COLUMN fact_order_payments.payment_sequential IS 'Degenerate dimension (sequence number of payment within order).';
COMMENT ON COLUMN fact_order_payments.purchase_date_key IS 'FK to dim_time — date the order was purchased.';
COMMENT ON COLUMN fact_order_payments.payment_installments IS 'Number of installments for this payment.';
COMMENT ON COLUMN fact_order_payments.payment_value IS 'Payment amount in BRL.';

-- Verify
SELECT COUNT(*) AS fact_order_payments_row_count FROM fact_order_payments;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;

-- ============================================================
-- Reconciliation: New Gold tables vs Silver sources
-- ============================================================
SELECT
  'fact_order_items' AS gold_table_name,
  (SELECT COUNT(*) FROM fact_order_items) AS gold_count,
  (SELECT COUNT(*) FROM `02-silver`.order_item) AS silver_count,
  (SELECT COUNT(*) FROM fact_order_items) - (SELECT COUNT(*) FROM `02-silver`.order_item) AS gold_minus_silver,
  CASE
    WHEN (SELECT COUNT(*) FROM fact_order_items) = (SELECT COUNT(*) FROM `02-silver`.order_item) THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS reconciliation_status
UNION ALL
SELECT
  'fact_order_payments' AS gold_table_name,
  (SELECT COUNT(*) FROM fact_order_payments) AS gold_count,
  (SELECT COUNT(*) FROM `02-silver`.order_payment) AS silver_count,
  (SELECT COUNT(*) FROM fact_order_payments) - (SELECT COUNT(*) FROM `02-silver`.order_payment) AS gold_minus_silver,
  CASE
    WHEN (SELECT COUNT(*) FROM fact_order_payments) = (SELECT COUNT(*) FROM `02-silver`.order_payment) THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS reconciliation_status
UNION ALL
SELECT
  'dim_product' AS gold_table_name,
  (SELECT COUNT(*) FROM dim_product) AS gold_count,
  (SELECT COUNT(*) FROM `02-silver`.product) AS silver_count,
  (SELECT COUNT(*) FROM dim_product) - (SELECT COUNT(*) FROM `02-silver`.product) AS gold_minus_silver,
  CASE
    WHEN (SELECT COUNT(*) FROM dim_product) = (SELECT COUNT(*) FROM `02-silver`.product) THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS reconciliation_status
UNION ALL
SELECT
  'dim_payment_type' AS gold_table_name,
  (SELECT COUNT(*) FROM dim_payment_type) AS gold_count,
  (SELECT COUNT(*) FROM `02-silver`.payment) AS silver_count,
  (SELECT COUNT(*) FROM dim_payment_type) - (SELECT COUNT(*) FROM `02-silver`.payment) AS gold_minus_silver,
  CASE
    WHEN (SELECT COUNT(*) FROM dim_payment_type) = (SELECT COUNT(*) FROM `02-silver`.payment) THEN 'MATCH'
    ELSE 'MISMATCH'
  END AS reconciliation_status;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `03-gold`;

-- ============================================================
-- Cross-fact query: Monthly revenue by item value vs payment value
-- Demonstrates conformed dim_time across fact_order_items and fact_order_payments
-- ============================================================
WITH item_revenue AS (
  SELECT
    dt.year_month,
    SUM(foi.total_value) AS total_item_value,
    COUNT(*) AS item_count
  FROM fact_order_items foi
  JOIN dim_time dt ON foi.purchase_date_key = dt.date_key
  GROUP BY dt.year_month
),
payment_revenue AS (
  SELECT
    dt.year_month,
    SUM(fop.payment_value) AS total_payment_value,
    COUNT(*) AS payment_count
  FROM fact_order_payments fop
  JOIN dim_time dt ON fop.purchase_date_key = dt.date_key
  GROUP BY dt.year_month
)
SELECT
  COALESCE(ir.year_month, pr.year_month) AS year_month,
  ir.total_item_value,
  ir.item_count,
  pr.total_payment_value,
  pr.payment_count,
  ROUND(pr.total_payment_value - ir.total_item_value, 2) AS payment_minus_item_value
FROM item_revenue ir
FULL OUTER JOIN payment_revenue pr ON ir.year_month = pr.year_month
ORDER BY year_month;